# Depth Pro on C3VD (zero-shot, raw frames)

Model: `apple/DepthPro-hf`, used as released (no training or fine-tuning). Frames go in as stored; only Depth Pro's own preprocessing is applied. Code: `results/E/run_depthpro.py`. Dev = trans_t1_a, test = trans_t2_b.

**Methods:** Raw metric = output in mm, nothing fitted; Dev scale, Linear and Smooth are fitted on development data; Oracle rows use each frame's ground truth.

**Masks:** Mask A (main table) = 0 < raw < 65535 and RGB max > 10. Mask B (second table) = raw > 0.

This notebook shows the saved tables and does not rerun the test set, which is scored once.

In [1]:
import json
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
SPLIT = 'test'   # 'dev' or 'test'
summary = pd.read_csv(HERE / f'depthpro_{SPLIT}_summary.csv')
cols = {'method': 'Method', 'absrel': 'AbsRel ↓', 'rmse': 'RMSE (mm) ↓', 'mae': 'MAE (mm) ↓', 'delta1': 'δ<1.25 ↑'}
for mk in ['A', 'B']:
    t = summary[summary['mask'] == mk][list(cols)].rename(columns=cols)
    print(f'{SPLIT}, mask {mk}')
    print(t.round(3).to_string(index=False), end='\n\n')

test, mask A
         Method  AbsRel ↓  RMSE (mm) ↓  MAE (mm) ↓  δ<1.25 ↑
Constant median     0.550       17.710      11.454     0.275
     Raw metric    19.624      318.861     311.652     0.000
      Dev scale     0.586       13.532       9.693     0.308
         Linear     0.875       14.329      11.729     0.269
         Smooth     0.826       14.085      11.282     0.286
  Oracle median     0.468       14.236       9.315     0.346
     Oracle LSQ     0.251        4.930       3.747     0.637

test, mask B
         Method  AbsRel ↓  RMSE (mm) ↓  MAE (mm) ↓  δ<1.25 ↑
Constant median     0.565       21.209      12.976     0.270
     Raw metric    19.305      323.201     315.118     0.000
      Dev scale     0.601       16.107      10.866     0.298
         Linear     0.933       15.946      13.039     0.249
         Smooth     0.868       15.805      12.466     0.270
  Oracle median     0.480       17.105      10.529     0.341
     Oracle LSQ     0.253        5.210       3.954     0.6

In [2]:
info = json.load(open(HERE / f'run_info_{SPLIT}.json'))
print({k: info[k] for k in ['python', 'torch', 'transformers', 'device', 'seed', 'model', 'sec_per_frame', 'mean_pred_fov_deg']})

{'python': '3.11.17', 'torch': '2.13.0', 'transformers': '5.20.2', 'device': 'mps', 'seed': 42, 'model': 'apple/DepthPro-hf', 'sec_per_frame': 4.8543185232552855, 'mean_pred_fov_deg': 50.58616504854369}
